# AutoDQA — Bedrock + LangGraph + Cloudflare agent

A single ReAct agent for **Autonomous Data Quality Assessment**, wired to the real project resources:

- **Reasoning:** Claude on **Amazon Bedrock** (`ChatBedrockConverse`), always via the **Cloudflare AI Gateway** — the gateway holds the Bedrock credentials (BYOK) and signs the upstream requests
- **Orchestration:** **LangGraph** `create_react_agent` (reason -> act -> observe)
- **Tools:**
  - `query_cdw` - read-only T-SQL against the SQL Server CDW (`192.168.1.65 / CDW`), runs **locally** via pyodbc
  - `search_etl` / `read_etl_file` - search & read the ETL repo at `/home/atth/gitlab/cdw`, runs **locally**
  - `run_python` - arbitrary Python in a **Cloudflare Sandbox** (isolated compute on retrieved data)

**Why local vs. sandbox?** The CDW and ETL files live on this host/LAN; Cloudflare's cloud sandbox
can't reach them, so the I/O tools run locally (guard-railed: read-only SQL + repo-scoped reads).
The sandbox is where *untrusted computation* is isolated.

This is the **simple** starting point: one agent, one task (query the DB + read the ETL). It does
**not** run the full multi-phase DQA graph yet.

Run top to bottom. Connection details come from `config.yaml`; secrets (AI Gateway token, sandbox
secret) from `.env` — no AWS credentials needed locally. Note: at last check DEMOGRAPHIC had
~10k rows; ENCOUNTER/DIAGNOSIS were empty.

## Architecture (target — code-mode)

Rendered from the same Mermaid source as [`docs/target_architecture.md`](docs/target_architecture.md).
This is the **end-state** design: orchestration stays local while the DB query and ETL reads move
*inside* the in-network sandbox. The cells below still run the **current** setup (DB/ETL tools local,
sandbox for compute) — see the intro above for that distinction.

In [1]:
%pip install -q "langchain>=1.0" "langchain-aws>=1.4.5" "langgraph>=0.6" "langchain-core>=0.3" "boto3>=1.35.87" pyodbc pyyaml requests

Note: you may need to restart the kernel to use updated packages.


In [1]:
import os, json
from pathlib import Path
import yaml


def load_dotenv(path: Path) -> None:
    """Load KEY=VALUE / 'export KEY=VALUE' lines from .env into os.environ (existing env wins)."""
    if not path.exists():
        print(f"(no .env at {path})"); return
    for raw in path.read_text().splitlines():
        line = raw.strip().removeprefix("export ").strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        k, _, v = line.partition("=")
        os.environ.setdefault(k.strip(), v.strip().strip('"').strip("'"))


load_dotenv(Path.cwd() / ".env")

CONFIG = yaml.safe_load(open("config.yaml"))
CONN = CONFIG["connection"]
ETL_REPO = Path(CONFIG["sources"]["etl"]["path"])
TABLES = CONFIG.get("tables", [])

REGION = os.environ.get("AWS_DEFAULT_REGION", "us-east-1")
MODEL_ID = "us.anthropic.claude-sonnet-4-5-20250929-v1:0"

print("AI Gateway creds set:", all(k in os.environ for k in ("CF_ACCOUNT_ID", "CF_AIG_GATEWAY", "CF_AIG_TOKEN")))
print("Sandbox URL set:", "SANDBOX_WORKER_URL" in os.environ)
print("CDW:", CONN["server"], "/", CONN["database"], "| ETL repo:", ETL_REPO, "exists:", ETL_REPO.is_dir())

AI Gateway creds set: True
Sandbox URL set: True
CDW: 192.168.1.65 / CDW | ETL repo: /home/atth/gitlab/cdw exists: True


In [2]:
# --- Tool 1: query_cdw (LOCAL, read-only T-SQL against SQL Server) ---
import pyodbc
from langchain_core.tools import tool

FORBIDDEN = ["INSERT", "UPDATE", "DELETE", "DROP", "CREATE", "ALTER", "TRUNCATE",
             "EXEC", "EXECUTE", "MERGE", "GRANT", "REVOKE", "DENY", "sp_", "xp_",
             "OPENROWSET", "OPENQUERY", "OPENDATASOURCE", "BULK", "INTO",
             "DBCC", "BACKUP", "RESTORE", "SHUTDOWN"]
MAX_ROWS = 100
_conn = None


def _cdw():
    global _conn
    if _conn is not None:
        try:
            _conn.cursor().execute("SELECT 1"); return _conn
        except Exception:
            _conn = None
    cs = f"DRIVER={{{CONN['driver']}}};SERVER={CONN['server']};DATABASE={CONN['database']};"
    if CONN.get("uid"):
        cs += f"UID={CONN['uid']};PWD={CONN['pwd']};"
    else:
        cs += f"Trusted_Connection={CONN.get('trusted_connection', 'yes')};"
    if CONN.get("TrustServerCertificate"):
        cs += f"TrustServerCertificate={CONN['TrustServerCertificate']};"
    _conn = pyodbc.connect(cs, readonly=True); _conn.autocommit = True
    return _conn


def _ser(v):
    if v is None or isinstance(v, (int, float, str, bool)):
        return v
    if isinstance(v, bytes):
        return v.hex()
    return str(v)


def _check_readonly(q):
    norm = " " + " ".join(q.upper().split()) + " "
    for kw in FORBIDDEN:
        if f" {kw.upper()} " in norm:
            raise ValueError(f"Keyword '{kw}' not allowed - read-only access only.")


@tool
def query_cdw(sql: str) -> str:
    """Run a read-only T-SQL SELECT against the CDW (SQL Server) and return rows as JSON.

    Dialect is T-SQL: use TOP (not LIMIT). INSERT/UPDATE/DELETE/DDL are rejected.
    Results are capped at 100 rows - aggregate (COUNT, GROUP BY) for profiling.
    """
    try:
        _check_readonly(sql)
    except ValueError as e:
        return json.dumps({"error": str(e)})
    try:
        cur = _cdw().cursor(); cur.execute(sql)
        cols = [d[0] for d in cur.description] if cur.description else []
        rows = [dict(zip(cols, [_ser(v) for v in r])) for r in cur.fetchmany(MAX_ROWS)]
        out = {"columns": cols, "row_count": len(rows), "rows": rows}
        if len(rows) == MAX_ROWS:
            out["note"] = f"truncated at {MAX_ROWS} rows; add TOP/WHERE or aggregate"
        return json.dumps(out, default=str)
    except Exception as e:
        return json.dumps({"error": f"{type(e).__name__}: {e}"})


print("query_cdw ready")

query_cdw ready


In [3]:
# --- Tools 2 & 3: search_etl / read_etl_file (LOCAL, repo-scoped) ---
import subprocess


@tool
def search_etl(query: str, max_results: int = 40) -> str:
    """Search the ETL repo's .sql files for a string (case-insensitive grep).

    Returns matching file paths (relative to the repo), line numbers, and lines.
    Use this to find which views/procs/functions reference a table or column.
    """
    try:
        r = subprocess.run(["grep", "-rni", "--include", "*.sql", "-m", str(max_results),
                            query, str(ETL_REPO)], capture_output=True, text=True, timeout=30)
    except subprocess.TimeoutExpired:
        return json.dumps({"error": "search timed out"})
    matches = []
    for line in r.stdout.splitlines():
        if not line:
            continue
        parts = line.split(":", 2)
        if len(parts) >= 3:
            matches.append({"file": os.path.relpath(parts[0], str(ETL_REPO)),
                            "line": int(parts[1]), "content": parts[2].strip()})
    return json.dumps({"query": query, "match_count": len(matches), "matches": matches})


@tool
def read_etl_file(path: str, start_line: int = 0, max_lines: int = 400) -> str:
    """Read an ETL .sql file. `path` is relative to the ETL repo (or absolute within it).

    Returns up to max_lines lines of content plus metadata.
    """
    full = Path(path) if os.path.isabs(path) else ETL_REPO / path
    try:
        if not str(full.resolve()).startswith(str(ETL_REPO.resolve())):
            return json.dumps({"error": "path is outside the ETL repository"})
    except Exception:
        return json.dumps({"error": "invalid path"})
    if not full.is_file():
        return json.dumps({"error": f"file not found: {path}"})
    lines = full.read_text(encoding="utf-8", errors="replace").splitlines(keepends=True)
    sel = lines[start_line:start_line + max_lines]
    return json.dumps({"file": os.path.relpath(full, str(ETL_REPO)), "total_lines": len(lines),
                       "start_line": start_line, "lines_returned": len(sel),
                       "truncated": start_line + max_lines < len(lines), "content": "".join(sel)})


print("ETL tools ready; repo:", ETL_REPO, "exists:", ETL_REPO.is_dir())

ETL tools ready; repo: /home/atth/gitlab/cdw exists: True


In [4]:
# --- Tool 4: run_python (Cloudflare sandbox, isolated compute) ---
import requests

SANDBOX_URL = os.environ["SANDBOX_WORKER_URL"]
SANDBOX_SECRET = os.environ["SANDBOX_SHARED_SECRET"]


@tool
def run_python(code: str) -> str:
    """Execute Python in an isolated Cloudflare sandbox (remote, NOT this host).

    Use for computation/analysis on data you've already retrieved with query_cdw.
    Has no access to the CDW or the ETL repo. Print anything you want to see.
    """
    resp = requests.post(SANDBOX_URL, headers={"Authorization": f"Bearer {SANDBOX_SECRET}"},
                         json={"code": code, "language": "python", "session": "autodqa"}, timeout=120)
    resp.raise_for_status()
    data = resp.json()
    if data.get("error"):
        e = data["error"]
        return f"Error: {e.get('name')}: {e.get('value') or e.get('message')}"
    texts = [r["text"] for r in data.get("results", []) if r.get("text")]
    return "\n".join(filter(None, [data.get("stdout", ""), *texts])).strip() or "(no output)"


print("run_python ready")

run_python ready


In [5]:
# --- Build the agent ---
import boto3
from botocore import UNSIGNED
from botocore.config import Config
from langchain_aws import ChatBedrockConverse
from langchain.agents import create_agent

SYSTEM = (
    "You are an AutoDQA analyst agent for a PCORnet CDM clinical data warehouse on SQL Server. "
    "Tools: query_cdw (read-only T-SQL against the CDW - use TOP, not LIMIT); "
    "search_etl and read_etl_file (search/read the ETL SQL codebase); run_python "
    "(isolated Cloudflare sandbox for computation on data you have already pulled). "
    "Work step by step: gather evidence with the tools, then answer concisely with concrete "
    "numbers, and cite ETL file paths and line numbers when you reference code. "
    "Never attempt to modify data."
)

# Bedrock calls route through the Cloudflare AI Gateway (observability, control,
# Guardrails, and BYOK credential centralization). The gateway holds the IAM SigV4
# keys and signs the upstream request, so no AWS credentials are needed locally —
# auth here is just the cf-aig-authorization header (CF_AIG_TOKEN).
# Verified working 2026-06-03 via BYOK (IAM SigV4 keys stored in the gateway).
# Requires CF_ACCOUNT_ID, CF_AIG_GATEWAY, CF_AIG_TOKEN in .env (loaded in cell 2).
# NOTE: the gateway logs request/response bodies by default — disable body logging
# (or confirm a compliant/BAA boundary) before sending PHI through it.
acct = os.environ["CF_ACCOUNT_ID"]
gw = os.environ["CF_AIG_GATEWAY"]
token = os.environ["CF_AIG_TOKEN"]
# Bedrock-runtime endpoint on the gateway; boto3 appends /model/<id>/converse.
aig_endpoint = f"https://gateway.ai.cloudflare.com/v1/{acct}/{gw}/aws-bedrock/bedrock-runtime/{REGION}"

bedrock_client = boto3.client(
    "bedrock-runtime",
    region_name=REGION,
    endpoint_url=aig_endpoint,
    config=Config(signature_version=UNSIGNED),  # BYOK: the gateway does the SigV4 signing
)


def _add_aig_auth(request, **kwargs):
    request.headers["cf-aig-authorization"] = f"Bearer {token}"


bedrock_client.meta.events.register("before-send.bedrock-runtime", _add_aig_auth)

llm = ChatBedrockConverse(model=MODEL_ID, client=bedrock_client, region_name=REGION, temperature=0)
print("LLM -> Cloudflare AI Gateway ->", aig_endpoint)

TOOLS = [query_cdw, search_etl, read_etl_file, run_python]
agent = create_agent(model=llm, tools=TOOLS, system_prompt=SYSTEM)
print("agent built with tools:", [t.name for t in TOOLS])

LLM -> Cloudflare AI Gateway -> https://gateway.ai.cloudflare.com/v1/ccfb16c028dc5731cab073bdf569d8de/data-curation-gateway/aws-bedrock/bedrock-runtime/us-east-1
agent built with tools: ['query_cdw', 'search_etl', 'read_etl_file', 'run_python']


## Run the agent - simple task

Query the database and read the ETL. Change `task` and re-run. `stream_mode="values"` prints the
full message list after each step, so you can watch the tool calls and results.

In [7]:
task = ("How many rows are in the DEMOGRAPHIC table? Then find which ETL view(s) load "
        "DEMOGRAPHIC and briefly summarize the transformations they apply.")

print(f">>> {task}\n")
for chunk in agent.stream(
    {"messages": [("user", task)]},
    stream_mode="values",
):
    chunk["messages"][-1].pretty_print()

>>> How many rows are in the DEMOGRAPHIC table? Then find which ETL view(s) load DEMOGRAPHIC and briefly summarize the transformations they apply.

================================ Human Message =================================

How many rows are in the DEMOGRAPHIC table? Then find which ETL view(s) load DEMOGRAPHIC and briefly summarize the transformations they apply.
================================== Ai Message ==================================

[{'type': 'text', 'text': "I'll help you find the row count in the DEMOGRAPHIC table and identify the ETL processes that load it."}, {'type': 'tool_use', 'name': 'query_cdw', 'input': {'sql': 'SELECT COUNT(*) as row_count FROM DEMOGRAPHIC'}, 'id': 'tooluse_MckHS0bDZqc13T5N3P08ta'}, {'type': 'tool_use', 'name': 'search_etl', 'input': {'query': 'INSERT INTO DEMOGRAPHIC'}, 'id': 'tooluse_XS9K9fyRoCn2DU9MW0EarQ'}]
Tool Calls:
  query_cdw (tooluse_MckHS0bDZqc13T5N3P08ta)
 Call ID: tooluse_MckHS0bDZqc13T5N3P08ta
  Args:
    sql: SELECT COUNT(*) 

================================== Ai Message ==================================

[{'type': 'text', 'text': 'Now let me search for the main ETL view that loads DEMOGRAPHIC:'}, {'type': 'tool_use', 'name': 'read_etl_file', 'input': {'path': 'etl/procedures/etl.load_DEMOGRAPHIC.StoredProcedure.sql', 'max_lines': 100}, 'id': 'tooluse_bCOXouW8n6sBZSysZ9nopc'}]
Tool Calls:
  read_etl_file (tooluse_bCOXouW8n6sBZSysZ9nopc)
 Call ID: tooluse_bCOXouW8n6sBZSysZ9nopc
  Args:
    path: etl/procedures/etl.load_DEMOGRAPHIC.StoredProcedure.sql
    max_lines: 100
================================= Tool Message =================================
Name: read_etl_file

{"file": "etl/procedures/etl.load_DEMOGRAPHIC.StoredProcedure.sql", "total_lines": 486, "start_line": 0, "lines_returned": 100, "truncated": true, "content": "SET ANSI_NULLS ON\nGO\n\nSET QUOTED_IDENTIFIER ON\nGO\n\n/******************************************************************************\n  File:         load_demographic.sql\n  Object:

================================== Ai Message ==================================

[{'type': 'text', 'text': 'Now let me look at the etl.DEMOGRAPHIC view that is the source:'}, {'type': 'tool_use', 'name': 'read_etl_file', 'input': {'path': 'etl/tables/etl.DEMOGRAPHIC.View.sql'}, 'id': 'tooluse_aJCB99lCrhpGUfFKg9ihCf'}]
Tool Calls:
  read_etl_file (tooluse_aJCB99lCrhpGUfFKg9ihCf)
 Call ID: tooluse_aJCB99lCrhpGUfFKg9ihCf
  Args:
    path: etl/tables/etl.DEMOGRAPHIC.View.sql
================================= Tool Message =================================
Name: read_etl_file

{"file": "etl/tables/etl.DEMOGRAPHIC.View.sql", "total_lines": 65, "start_line": 0, "lines_returned": 65, "truncated": false, "content": "--USE [CDW_Test] -- Should be agnostic and applied through the Jenkins script: https://sbpmtatlv001.uthouston.edu/wp/1821\n--GO\n\n/****** Object:  View [etl].[DEMOGRAPHIC]    Script Date: 3/15/2024 12:03:34 PM ******/\n-- Removed GO parts from script (#1867) and changed from CREATE

================================== Ai Message ==================================

Perfect! Now I have enough information to provide a comprehensive answer. Let me create a summary:

## Summary

**Row Count:** The DEMOGRAPHIC table contains **10,000 rows**.

**ETL Process:** The DEMOGRAPHIC table is loaded by the stored procedure **`etl.load_demographic`** (located in `etl/procedures/etl.load_DEMOGRAPHIC.StoredProcedure.sql`).

### Key Transformations Applied:

1. **Source Data Integration** (lines 283-294 in `etl.load_DEMOGRAPHIC.StoredProcedure.sql`):
   - Pulls from the unified view `etl.DEMOGRAPHIC` which consolidates data from the Master Patient Index (MPI)
   - The main source view (`etl/tables/etl.DEMOGRAPHIC.View.sql`) joins `MasterPatientIndex.dbo.EnterpriseRecords` with `EnterpriseRecords_Ext`

2. **Patient Cohort Filtering** (line 294):
   - Only includes patients in `etl.CDW_PATIENT_COHORT` (built from appointments, encounters, and MPI records)
   - Ensures only active/relev